# LAB- P5: El Gobierno y la Política Fiscal (Julia)
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de Práctica**: LAB-P5-v1.0
*   **Capítulo de Referencia**: Capítulo 6, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Explorar el papel macroeconómico y distorsionador del sector público en una economía con agentes racionales de ciclo de vida finito. Estudiaremos:
    1. Impuestos no distorsionadores de suma fija (lump-sum) y la demostración computacional del **Principio de Equivalencia Ricardiana**.
    2. Impuestos distorsionadores directos e indirectos (sobre el consumo $\tau^c$, el trabajo $\tau^w$ y las rentas del capital $\tau^r$), y sus efectos sobre la asignación del tiempo y el ahorro.
    3. Un sistema de Seguridad Social de capitalización (ahorro forzoso) y su efecto de sustitución perfecta sobre el ahorro privado voluntario.

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Analizar** la diferencia matemática y económica entre impuestos de suma fija (no distorsionadores) e impuestos distorsionadores.
2.  **Comprender** y verificar computacionalmente el Principio de Equivalencia Ricardiana en el comportamiento del consumo.
3.  **Simular** el impacto de las distorsiones impositivas sobre la oferta de trabajo y la trayectoria intertemporal de consumo y ahorro.
4.  **Modelar** un sistema de Seguridad Social de capitalización y comprender la sustitución perfecta entre ahorro forzoso y voluntario.
5.  **Resolver** de forma exacta equilibria descentralizados con impuestos empleando el resolvedor de FOCs (`fsolve`) y la optimización convexa equivalente (`solve_direct_optim`).


> **👋 BIENVENIDA A LA PRÁCTICA - LEER ANTES DE EMPEZAR**
> 
> *   **¿Nunca has usado Jupyter?** No te preocupes. Este cuaderno es interactivo. Haz clic en cualquier celda de código y pulsa **`Shift + Enter`** para ejecutarla. Ve de arriba a abajo en orden.
> *   **¿Se ha congelado o sale un asterisco `[*]` eterno?** Ve al menú superior y dale a `Kernel` ➔ `Restart`.
> *   **El objetivo** de esta práctica es que juegues con la economía. Cambia los números del código que representan impuestos, dinero o tecnología, vuelve a ejecutar y mira los gráficos. ¡No puedes romper nada!
>

### 🕹️ GUÍA RÁPIDA PARA DUMMIES - Gobierno y Política Fiscal
*   **¿Qué estamos haciendo aquí?** Estudiando cómo afectan los impuestos del gobierno a las decisiones de las personas.
*   **Impuestos Distorsionadores:** Cuando el gobierno cobra impuestos sobre el trabajo, la gente decide trabajar menos porque se queda con menos dinero neto.
*   **¡Prueba esto!** Incrementa la tasa impositiva (los impuestos) en el modelo y observa la caída en el consumo y en las horas de trabajo.


In [ ]:
# Este cuaderno depende del paquete `MacroAIComp` (Project.toml/Manifest.toml
# en la raíz del repositorio). En MyBinder (ver docs/DESPLIEGUE_BINDER.md) y en
# tu entorno local, el kernel ya arranca dentro del repositorio clonado, así
# que la celda siguiente activa e instancia el proyecto automáticamente.
# Nota: Google Colab no soporta Julia de forma nativa desde un notebook .ipynb;
# para la versión Julia de esta práctica usa MyBinder.


In [ ]:
using Pkg
Pkg.activate("../..")
Pkg.instantiate()

using MacroAIComp
using Plots
import Plots: mm
default(gridalpha=0.6, gridstyle=:dot)  # estilo de grid consistente con la versión Python
using LinearAlgebra
using NLsolve
using Optim
using Interact
using BenchmarkTools


## 1. Impuestos de Suma Fija (Lump-Sum) y Equivalencia Ricardiana

Un impuesto es **no distorsionador** (lump-sum) si la carga impositiva es fija e independiente de las decisiones de los agentes. En este caso, el impuesto de ingresos exógenos es equivalente a un impuesto de suma fija porque el agente no puede alterar su base imponible (el salario es exógeno y no hay elección de ocio).

El problema del agente consiste en maximizar:
$$\max_{\{C_t\}_{t=0}^{T-1}} \sum_{t=0}^{T-1} \beta^t \ln(C_t)$$

Sujeto a la restricción presupuestaria secuencial:
$$C_t + B_t = (1-\tau^w_t) W_t + (1+R)B_{t-1} + G_t$$

Donde $\tau^w_t$ es la tasa impositiva y $G_t$ son las transferencias del gobierno.

### 1.1 El Principio de Equivalencia Ricardiana
Si el gobierno financia un gasto público devolviendo toda la recaudación de impuestos al consumidor mediante transferencias de suma fija ($G_t = \tau^w_t W_t$), la restricción presupuestaria en equilibrio se reduce a la de una economía libre de impuestos:
$$C_t + B_t = W_t + (1+R)B_{t-1}$$

En este caso, la trayectoria óptima de consumo y ahorro es **completamente insensible** a la tasa impositiva. Si las transferencias no se devuelven ($G_t = 0$), se produce un efecto renta puro: los agentes consumen y ahorran menos, pero la pendiente del consumo (la ecuación de Euler) permanece idéntica.


In [ ]:
params_lumpsum = default_calibration(FiscalPolicyParameters)

# Glosario didáctico: descripción económica y símbolo de cada parámetro técnico
descriptions = Dict(
    "T" => "Duración del ciclo de vida [T]",
    "beta" => "Factor de descuento intertemporal [β]",
    "R" => "Tipo de interés real [R]",
    "gamma" => "Peso del consumo en la función de utilidad [γ]",
    "B0" => "Activos iniciales [B0]",
    "tauw" => "Tasa impositiva sobre el salario [τw]",
    "tauc" => "Tasa impositiva sobre el consumo [τc]",
    "taur" => "Tasa impositiva sobre las rentas del capital [τr]",
    "tau_ss" => "Cotización a la Seguridad Social [τss]",
    "t_star" => "Periodo de jubilación [t*]",
)

println("CALIBRACIÓN ECONÓMICA DE REFERENCIA (Valores base del Libro):")
println("="^75)
println(rpad("Variable", 12), " | ", rpad("Valor", 6), " | ", rpad("Descripción Económica", 50))
println("-"^75)
for field in fieldnames(typeof(params_lumpsum))
    name = string(field)
    value = getfield(params_lumpsum, field)
    desc = get(descriptions, name, "Parámetro del modelo")
    println("  ", rpad(name, 10), " | ", rpad(value, 6), " | ", rpad(desc, 50))
end
println("="^75)


In [ ]:
# Generar salario constante (calibración base: T=30, beta=0.97, R=0.05)
W = fill(10.0, params_lumpsum.T)

# Caso con impuesto de suma fija (tauw=0.40) y devolución de la recaudación (G=T)
params_tax = FiscalPolicyParameters(30, 0.97, 0.05, params_lumpsum.gamma, 0.0, 0.40, 0.0, 0.0, 0.0, 26)

# 1. Resolver caso base sin impuestos
params_no_tax = FiscalPolicyParameters(30, 0.97, 0.05, params_lumpsum.gamma, 0.0, 0.0, 0.0, 0.0, 0.0, 26)
res_base = solve_non_distortionary(params_no_tax, W)

# 2. Resolver con impuestos de suma fija y devolución (Equivalencia Ricardiana)
res_tax = solve_non_distortionary(params_tax, W, true)

println("Diferencia media en Consumo (Equivalencia Ricardiana): ", sum(abs.(res_base["C"] .- res_tax["C"])) / params_lumpsum.T)


## 2. Impuestos Distorsionadores (Consumo, Trabajo y Ahorro)

Cuando la oferta de trabajo es endógena (decisión intratemporal entre consumo y ocio) y el ahorro genera rentas financieras, las tasas impositivas sobre el consumo ($\tau^c$), el salario ($\tau^w$) y los rendimientos financieros ($\tau^r$) alteran los precios relativos.

El problema del agente consiste en maximizar:
$$\max_{\{C_t, L_t\}_{t=0}^{T-1}} \sum_{t=0}^{T-1} \beta^t \left[ \gamma \ln(C_t) + (1-\gamma) \ln(1 - L_t) \right]$$

Sujeto a la restricción presupuestaria distorsionada:
$$(1+\tau^c_t) C_t + B_t = (1-\tau^w_t) W_t L_t + [1 + (1-\tau^r_t) R] B_{t-1} + G_t$$

### 2.1 Canal de Distorsión
Las distorsiones de la política fiscal actúan directamente a través de las condiciones marginales:
1.  **Distorsión Intratemporal (Oferta de Trabajo):**
    $$(1-\tau^w_t) W_t (1-L_t) = \frac{1-\gamma}{\gamma} (1+\tau^c_t) C_t$$
    Un aumento de $\tau^w$ (impuesto al salario) o $\tau^c$ (impuesto al consumo) reduce la rentabilidad marginal de trabajar frente al ocio, desincentivando la oferta de trabajo ($L_t$ cae).
2.  **Distorsión Intertemporal (Ahorro):**
    $$(1+\tau^c_{t+1}) C_{t+1} = \beta [1 + (1-\tau^r_t)R] (1+\tau^c_t) C_t$$
    El impuesto sobre el capital ($\tau^r$) reduce el rendimiento neto del ahorro, lo que hace al consumidor más propenso a consumir hoy que mañana (la trayectoria de consumo se aplana y el ahorro cae).


In [ ]:
# Simulación interactiva con Interact.jl (Impuestos Distorsionadores)
@manipulate for tauc_val in 0.0:0.05:0.50, tauw_val in 0.0:0.05:0.50, taur_val in 0.0:0.05:0.80, ret_opt in ["lump_sum", "government_spending"]

    is_lump_sum_return = (ret_opt == "lump_sum")
    params = FiscalPolicyParameters(30, 0.97, 0.05, 0.40, 0.0, tauw_val, tauc_val, taur_val, 0.0, 0)
    W_sim = fill(100.0, params.T)
    res = solve_distortionary_foc(params, W_sim, is_lump_sum_return)
    
    t_axis = 0:(params.T - 1)
    
    # Panel 1: Consumo e Ingresos
    p1 = plot(t_axis, res["C"], color="#7A3E9F", lw=2.5, label="Consumo (C)")
    plot!(t_axis, res["W_L"], color="#8EAD3A", lw=2.5, ls=:dash, label="Ingreso Neto")
    title!("Consumo de Bienes")
    xlabel!("Periodo (t)")
    ylabel!("Unidades de C")

    # Panel 2: Oferta de Trabajo y Ocio
    p2 = plot(t_axis, res["L"], color="#E05A47", lw=2.5, label="Trabajo (L)")
    plot!(t_axis, res["O"], color="#3BB193", lw=2.5, ls=:dot, label="Ocio (O=1-L)")
    ylims!(-0.05, 1.05)
    title!("Oferta de Trabajo (L)")
    xlabel!("Periodo (t)")
    ylabel!("Fracción de Tiempo Trabajado")

    # Panel 3: Activos
    p3 = plot(t_axis, res["B"], color="#004C97", lw=2.5, label="Activos (B)")
    hline!([0.0], color=:black, ls=:dot, label="")
    plot!(t_axis, max.(res["B"], 0.0), fillrange=0, fillalpha=0.2, color="#004C97", lw=0, label="Ahorro")
    plot!(t_axis, min.(res["B"], 0.0), fillrange=0, fillalpha=0.2, color="#D95319", lw=0, label="Deuda")
    title!("Acumulación de Activos")
    xlabel!("Periodo (t)")
    ylabel!("Activos (B)")
    
    plot(p1, p2, p3, layout=(1,3), size=(1100, 350), 
         plot_title="Efecto de Impuestos Distorsionadores", top_margin=10mm)
end


## 3. El Sistema de Seguridad Social de Capitalización

En un sistema de Seguridad Social de capitalización, los trabajadores contribuyen obligatoriamente con una fracción $\tau^{ss}$ de su salario durante su periodo activo ($t < t^*$). Estas contribuciones se acumulan en un fondo de pensiones $D_t$ que obtiene rentabilidad a la tasa de interés de mercado $R$.

Al jubilarse ($t = t^*$), el individuo recibe el fondo acumulado como un pago único ($Pension$).

### 3.1 Sustitución Perfecta de Ahorro
En este modelo, el agente tiene previsión perfecta y libre acceso al mercado financiero (sin restricciones de liquidez). Por lo tanto, el ahorro forzoso de la Seguridad Social es un **sustituto perfecto** del ahorro privado voluntario.
Si el gobierno aumenta la tasa impositiva $\tau^{ss}$, el consumo óptimo del agente **no varía**. En su lugar, el agente reduce proporcionalmente su ahorro privado voluntario (llegando a ser negativo si es necesario tomar prestado contra el fondo de jubilación bloqueado) para mantener la misma trayectoria de consumo.


In [ ]:
# Simulación interactiva con Interact.jl (Seguridad Social)
@manipulate for tau_ss_val in 0.0:0.05:0.60, t_star_val in 15:1:29

    params_ss = FiscalPolicyParameters(30, 0.97, 0.05, 0.5, 0.0, 0.0, 0.0, 0.0, tau_ss_val, t_star_val)
    # Perfil salarial creciente: W_t = 10 + t durante la vida activa, 0 después
    W_ss = zeros(params_ss.T)
    W_ss[1:params_ss.t_star] .= 10.0 .+ (0:(params_ss.t_star - 1))

    res = solve_social_security(params_ss, W_ss)
    
    t_axis = 0:(params_ss.T - 1)
    
    # Panel 1: Consumo e Ingresos Laborales
    p1 = plot(t_axis, res["C"], color="#7A3E9F", lw=2.5, label="Consumo Óptimo")
    plot!(t_axis, W_ss, color=:gray, lw=2.0, ls=:dash, label="Salario Bruto (W)")
    plot!(t_axis, res["W_net"], color="#8EAD3A", lw=2.5, label="Ingreso Disponible")
    title!("Decisión de Consumo Óptimo")
    xlabel!("Periodo (t)")
    ylabel!("Consumo (C)")

    # Panel 2: Activos
    p2 = plot(t_axis, res["B"], color="#D95319", lw=2.5, label="Ahorro Privado (B)")
    plot!(t_axis, res["B_ss"], color="#8EAD3A", lw=2.5, label="Fondo SS (B_ss)")
    plot!(t_axis, res["B_total"], color=:black, lw=3.0, ls=:dashdot, label="Riqueza Total")
    hline!([0.0], color=:black, ls=:dot, label="")
    title!("Evolución de Riqueza")
    xlabel!("Periodo (t)")
    
    plot(p1, p2, layout=(1,2), size=(800, 350), 
         plot_title="Impacto del Sistema de Seguridad Social", top_margin=10mm)
end


## 4. Cuaderno de Bitácora (Actividades para el Alumno)

Responde a las siguientes cuestiones tras interactuar con las simulaciones de política fiscal:

1.  **Análisis de Equivalencia Ricardiana**:
    *   Activa la casilla `Devolver recaudación` en la simulación 1. Cambia el impuesto $\tau^w$ de 0 a 0.60. ¿Qué le ocurre al consumo en el panel de la izquierda? Explica detalladamente en base a la teoría microeconómica por qué no cambia.
    *   Desactiva la casilla `Devolver recaudación`. ¿Cómo responde el consumo y el ahorro? ¿Por qué la pendiente temporal del consumo sigue siendo positiva y con el mismo crecimiento relativo?
2.  **Efectos del Menú Fiscal Distorsionador**:
    *   En la simulación 2, establece `Equilibrio G = Recaudación` en `False`. Incrementa el impuesto sobre el trabajo $\tau^w$ de 0.0 a 0.50. ¿Cómo reacciona la oferta de trabajo $L_t$? ¿Por qué?
    *   Establece ahora un impuesto al consumo $\tau^c$ de 0.30 manteniendo $\tau^w = 0.0$. ¿Qué diferencia observas en el impacto sobre la oferta de trabajo respecto a $\tau^w$? ¿A qué se debe esto?
    *   Incrementa el impuesto al capital $\tau^r$ a 0.60. Observa el gráfico de Consumo. ¿Por qué la trayectoria se vuelve más plana (horizontal)? Explica el efecto intertemporal de la pérdida de interés real.
3.  **Seguridad Social e Implicaciones de Ahorro**:
    *   En la simulación 3, aumenta la cotización obligatoria $\tau^{ss}$ de 0.0 a 0.45. ¿Cambia el consumo óptimo del agente?
    *   Describe qué le ocurre a los activos privados (Panel 2) durante el periodo de vida laboral activa (antes de $t^* = 26$). ¿Por qué los activos privados caen a niveles fuertemente negativos (deuda)? Explica la lógica de "sustitución de ahorro" bajo previsión perfecta.


## 5. Buenas Prácticas Aplicadas en este Laboratorio
1.  **Aislamiento Paramétrico**: Las rutinas matemáticas y de simulación están completamente desacopladas de la interfaz visual, residiendo en `src/macroaicomp/models/fiscal_policy.jl`.
2.  **Modelización Equivalente de Equilibria**: La resolución de solve_direct_optim para el caso de transferencias devueltas ha sido implementada mediante un modelo equivalente surrogate en un solo paso, optimizando estabilidad numérica y eficiencia de cálculo.
3.  **Control de Versiones Limpio**: Este cuaderno ha sido limpiado de metadatos volátiles mediante `nbstripout` antes de guardarse en el repositorio.


## 6. Benchmark de Rendimiento (Fase III)
Evaluamos la velocidad de simulación usando `BenchmarkTools.jl`.

In [ ]:
# Benchmark simulation
@btime solve_distortionary_foc($params_lumpsum, fill(30.0, $params_lumpsum.T), true)
